# Blockchain Payments — Stage 3
## Geopolitical Event Study: USDC Payment Flows & Crypto Prices

**Author:** Saki Cansev  
**GitHub:** github.com/sakicansev  
**Date:** April 2026

---

### Research Question

Did USDC payment volumes on Ethereum change during the Iran–Israel–USA conflict escalation events documented in the companion [crypto-geopolitical-analysis](https://github.com/sakicansev/crypto-geopolitical-analysis) project — and do those changes correlate with BTC and ETH price reactions?

### Data Sources

- **On-chain:** Daily USDC transfer volume on Ethereum mainnet, October 2023 – April 2026, retrieved via Dune Analytics Query 10 (Query ID: 7365284) using `get_latest_result_dataframe()` and saved locally as CSV.
- **Off-chain:** Daily BTC and ETH closing prices from the `crypto_geopolitical.db` SQLite database built in the companion geopolitical analysis project.

### Methodology

Standard event study methodology (Brown & Warner, 1985): for each of the nine conflict escalation events, percentage changes in USDC volume, BTC price, and ETH price are calculated over D+1, D+3, and D+7 windows from the event date.

### Connection to Companion Project

The crypto-geopolitical-analysis project measured **price reactions** to conflict events. This notebook measures **payment behavior reactions**. Together they constitute a multi-asset, multi-metric event study — the first systematic analysis of geopolitical shocks on both crypto prices and on-chain payment flows simultaneously.

---

## Setup

### Terminal (run once before opening notebook)

Install all required libraries in one command:

```bash
pip install dune-client pandas numpy matplotlib seaborn scipy python-dotenv
```

### Environment File

Create a `.env` file in your stage 3 folder:

```bash
# In Terminal:
cd /path/to/blockchain-payments-learning/stage\ 3
touch .env
nano .env
# Type: DUNE_API_KEY=your_key_here
# Press Control+X, then Y, then Enter to save
echo ".env" >> .gitignore
```

The API key is found under **Connect → API keys** in the left sidebar of dune.com.

## Cell 1 — Imports and Configuration

All imports in one cell. Run this first every time you open the notebook.
Note: `pip install` commands go in Terminal, not here.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('All imports successful.')

All imports successful.


All imports successful.

## Cell 2 — Load API Key

Load the Dune API key from the `.env` file. Never paste the key directly into the notebook — that would expose it if the notebook is committed to GitHub.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
DUNE_API_KEY = os.getenv('DUNE_API_KEY')

# Verify the key loaded correctly — should print a long string, not None
print(repr(DUNE_API_KEY))

'0cdwzDVCwupQW4pnMSemLVqvgsg2r9Uu'


## Cell 3 — Retrieve USDC Data from Dune API

Retrieve Query 10 results using `get_latest_result_dataframe()` — this fetches the most recent cached execution result without triggering a new query run.

**Important:** `run_query_dataframe()` triggers a fresh execution and returns a 400 error on the free plan. Use `get_latest_result_dataframe()` instead — it retrieves the existing result from the last time you ran the query in the browser.

The API key location on Dune: **Connect → API keys** in the left sidebar (not in account settings).

Query 10 URL: https://dune.com/queries/7365284  
Query ID: **7365284**

In [3]:
from dune_client.client import DuneClient
from dune_client.query import QueryBase

client = DuneClient(api_key=DUNE_API_KEY)

# get_latest_result_dataframe() fetches cached results — no credit consumption
# Do NOT use run_query_dataframe() — triggers fresh execution, returns 400 on free plan
results = client.get_latest_result_dataframe(7365284)

print(results.head())
print(results.dtypes)

# Expected output:
# 936 rows covering October 2023 through April 2026
# Columns: date (object), transfer_count (int64), usdc_volume (float64)
# date is object at this stage — will be parsed in the next cell

                          date  transfer_count   usdc_volume
0  2023-10-01 00:00:00.000 UTC           42852  2.698518e+09
1  2023-10-02 00:00:00.000 UTC           61516  7.903753e+09
2  2023-10-03 00:00:00.000 UTC           50597  6.157389e+09
3  2023-10-04 00:00:00.000 UTC           56845  5.877146e+09
4  2023-10-05 00:00:00.000 UTC           54382  5.200415e+09
date               object
transfer_count      int64
usdc_volume       float64
dtype: object


## Cell 4 — Save to CSV

Save the API result immediately to a local CSV. This means the API is only called once — all subsequent work reads from the saved file. This avoids hitting rate limits and makes the notebook reproducible without an API connection.

In [ ]:
results.to_csv('usdc_daily_volumes.csv', index=False)
print('Saved.')

## Cell 5 — Load and Parse USDC Data

Load from the saved CSV. `parse_dates=['date']` converts the date column from string to `datetime64` automatically. `dt.tz_localize(None)` removes the UTC timezone offset — necessary for merging with the price data which has no timezone.

In [ ]:
df = pd.read_csv('usdc_daily_volumes.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'USDC data: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Date range: {df.date.min()} to {df.date.max()}')
print(df.dtypes)
print(df.head())

# Expected output:
# 936 rows — 2.5 years of daily data, Oct 2023 to Apr 2026
# date: datetime64[ns, UTC] — correctly parsed
# transfer_count: int64
# usdc_volume: float64 (in USD, already divided by 1e6 in the Dune query)

## Cell 6 — Load BTC and ETH Price Data from SQLite

The price data lives in the SQLite database from the companion crypto-geopolitical-analysis project. We connect to it directly at its original path — no need to copy it.

In [ ]:
import sqlite3

conn = sqlite3.connect('/Users/sakicansev/Documents/crypto-geopolitical-analysis/crypto_geopolitical.db')

# Verify available tables
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print('Tables:', tables)
# Expected: [('crypto_prices',), ('geopolitical_events',)]

prices = pd.read_sql(
    'SELECT Date, BTC_Close, ETH_Close FROM crypto_prices ORDER BY Date',
    conn,
    parse_dates=['Date']
)
prices['date'] = prices['Date'].dt.normalize()
conn.close()

print(f'Price data: {prices.shape[0]} days')
print(prices.head())

# Expected output:
# 917 rows — covers same period as USDC data
# Columns: Date, BTC_Close, ETH_Close, date
# BTC starts around $27,983 on 2023-10-01 — pre-conflict baseline

## Cell 7 — Merge Datasets

Merge USDC volume and price data on the `date` column.

**Important:** The USDC `date` column has UTC timezone (`datetime64[ns, UTC]`) while the price `date` has no timezone (`datetime64[ns]`). Pandas will raise a `ValueError` if you try to merge mismatched timezones. Fix: strip the timezone from the USDC column with `dt.tz_localize(None)` before merging.

The inner join keeps only dates present in both datasets. Since price data (917 days) is the smaller dataset, the merged result will have 917 rows.

In [ ]:
# Strip UTC timezone from USDC date before merging
df['date'] = df['date'].dt.tz_localize(None)

merged = pd.merge(
    df,
    prices[['date', 'BTC_Close', 'ETH_Close']],
    on='date',
    how='inner'
)

print(f'Merged dataset: {merged.shape[0]} rows, {merged.shape[1]} columns')
print(f'Columns: {merged.columns.tolist()}')
print(f'Null values:\n{merged.isnull().sum()}')
print(merged.head())

# Expected output:
# 917 rows, 5 columns: date, transfer_count, usdc_volume, BTC_Close, ETH_Close
# Zero null values — clean merge
# This is the primary analysis dataset for the entire event study

## Cell 8 — Verify October 7 Data Point

Before building the event study, verify the key finding from Stage 2 Query 10 is present in the merged dataset. The October 7, 2023 Hamas attack should show a sharp drop in USDC volume compared to the previous day.

In [ ]:
# Check the three days around the October 7 event
oct_window = merged[(merged['date'] >= '2023-10-05') & (merged['date'] <= '2023-10-09')]
print(oct_window[['date', 'transfer_count', 'usdc_volume', 'BTC_Close', 'ETH_Close']].to_string())

# Expected output:
# Oct 6: ~54,835 transfers, ~$6.67B volume
# Oct 7: ~38,209 transfers, ~$2.24B volume  <-- 66% drop
# Oct 8: ~37,698 transfers, ~$2.47B volume
#
# The drop is immediately visible — markets froze on the day of the attack.
# This confirms in Python the same finding first identified in the Dune browser query.

## Cell 9 — Save Merged Dataset

Save the merged dataset locally so it can be loaded directly in future sessions without re-running the API call and merge.

In [ ]:
merged.to_csv('merged_dataset.csv', index=False)
print('Saved.')

## Cell 10 — Define the Nine Conflict Events

The nine escalation events from the Iran–Israel–USA conflict documented in the companion geopolitical project. These are the exact same event dates used in that project's SQL analysis — enabling direct comparison between price reactions (measured there) and payment behavior reactions (measured here).

In [ ]:
events = [
    {'date': '2023-10-07', 'label': 'Hamas attacks Israel'},
    {'date': '2024-04-01', 'label': 'Israel strikes Iranian consulate, Damascus'},
    {'date': '2024-04-13', 'label': 'Iran launches 300+ drones at Israel'},
    {'date': '2024-04-19', 'label': 'Israel retaliates near Isfahan'},
    {'date': '2024-07-31', 'label': 'Assassination of Haniyeh in Tehran'},
    {'date': '2024-10-01', 'label': 'Iran: 180 ballistic missiles at Israel'},
    {'date': '2024-10-26', 'label': 'Israel largest direct strike on Iran'},
    {'date': '2025-06-13', 'label': 'Twelve-Day War begins'},
    {'date': '2026-02-28', 'label': 'US-Israel launch major strikes on Iran'},
]

events_df = pd.DataFrame(events)
events_df['date'] = pd.to_datetime(events_df['date'])

print(events_df)

# 9 events spanning October 2023 through February 2026
# Covers the full escalation arc: initial shock → regional escalation → direct war

## Cell 11 — Event Window Function

For each event, calculate percentage change in a variable from the event date over three windows: D+1 (day of event), D+3, and D+7.

Multiple windows allow us to distinguish:
- **D+1**: immediate shock response
- **D+3**: short-term adjustment
- **D+7**: whether the effect persisted or reversed

This is the standard event study window structure from financial economics (Brown & Warner, 1985).

In [ ]:
def event_window(df, event_date, column, windows=[1, 3, 7]):
    """
    Calculate percentage change from event_date over multiple windows.
    
    Parameters:
        df: merged DataFrame with date column
        event_date: the event date (datetime)
        column: variable to measure (usdc_volume, BTC_Close, ETH_Close)
        windows: list of day offsets to measure
    
    Returns:
        dict of {window_days: pct_change}
    """
    base_row = df[df['date'] == event_date]
    if base_row.empty:
        return {w: None for w in windows}
    base_val = base_row[column].values[0]

    results = {}
    for w in windows:
        target_date = event_date + pd.Timedelta(days=w)
        target_row = df[df['date'] == target_date]
        if target_row.empty:
            results[w] = None
        else:
            target_val = target_row[column].values[0]
            results[w] = round((target_val - base_val) / base_val * 100, 2)
    return results

print('event_window() function defined.')

# Quick test on October 7
test = event_window(merged, pd.Timestamp('2023-10-07'), 'usdc_volume')
print(f'Oct 7 USDC volume changes: {test}')
# Expected: D+1 should show large negative value confirming the 66% drop

## Cell 12 — Build the Full Event Study Table

Apply `event_window()` to all nine events and all three metrics. The result is a DataFrame with 9 rows (one per event) and 9 change columns (3 metrics × 3 windows).

In [ ]:
rows = []
for _, event in events_df.iterrows():
    edate = event['date']
    row = {'date': edate, 'event': event['label']}

    for col, prefix in [('usdc_volume', 'usdc'),
                        ('BTC_Close',   'btc'),
                        ('ETH_Close',   'eth')]:
        changes = event_window(merged, edate, col)
        for w, val in changes.items():
            row[f'{prefix}_d{w}'] = val

    rows.append(row)

event_study = pd.DataFrame(rows)

# Show the D+1 results for all events — the most immediate shock measure
print('Event Study — D+1 Percentage Changes:')
print(event_study[['date', 'event', 'usdc_d1', 'btc_d1', 'eth_d1']].to_string())

# What to look for in the results:
# - Oct 7, 2023: large negative USDC drop (confirming the 66% finding)
# - Jul 31, 2024 (Haniyeh assassination): expected largest BTC/ETH drops
#   per EMH — unanticipated events produce larger market reactions
# - Later events: watch for smaller reactions (desensitisation hypothesis)
# - Feb 28, 2026 (US-Israel strikes on Iran): may show positive USDC spike
#   — opposite direction from Oct 7, suggesting different market regime

## Chart 1 — USDC Volume Timeline with Event Markers

The full 2.5-year USDC daily volume series with vertical dashed lines at each of the nine conflict events. This chart gives immediate visual context for the entire dataset before the statistical analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(merged['date'],
                merged['usdc_volume'] / 1e9,
                alpha=0.3, color='steelblue')
ax.plot(merged['date'],
        merged['usdc_volume'] / 1e9,
        color='steelblue', linewidth=0.8)

for _, event in events_df.iterrows():
    ax.axvline(event['date'], color='crimson',
               linewidth=1.2, linestyle='--', alpha=0.7)

ax.set_title(
    'Daily USDC Transfer Volume on Ethereum\n'
    'with Iran–Israel–USA Conflict Events (Oct 2023 – Apr 2026)',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Date')
ax.set_ylabel('Volume (USD billions)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart1_usdc_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# What to look for:
# - Immediate trough at the first red line (Oct 7, 2023) — the 66% drop
# - Large spike near the last red line (Feb 2026, US-Israel strikes) — >$600B/week
# - General upward trend in baseline volume over the 2.5 year period
#   consistent with growing USDC adoption on Ethereum mainnet

## Chart 2 — Event Study Heatmap

Percentage changes for all three metrics (USDC volume, BTC price, ETH price) across all nine events and three time windows (D+1, D+3, D+7). Green = increase, red = decrease. This is the core output of the event study.

In [ ]:
cols = ['usdc_d1', 'usdc_d3', 'usdc_d7',
        'btc_d1',  'btc_d3',  'btc_d7',
        'eth_d1',  'eth_d3',  'eth_d7']

heatmap_data = event_study.set_index('event')[cols]

col_labels = [
    'USDC\nD+1', 'USDC\nD+3', 'USDC\nD+7',
    'BTC\nD+1',  'BTC\nD+3',  'BTC\nD+7',
    'ETH\nD+1',  'ETH\nD+3',  'ETH\nD+7'
]

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    heatmap_data.astype(float),
    annot=True, fmt='.1f', center=0,
    cmap='RdYlGn', linewidths=0.5,
    xticklabels=col_labels, ax=ax
)
ax.set_title(
    'Event Study: % Change in USDC Volume, BTC & ETH\n'
    'Following Iran–Israel–USA Conflict Escalation Events',
    fontsize=13, fontweight='bold'
)
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('chart2_event_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# What to look for:
# - Do USDC and price columns show the same colour direction? (correlated behavior)
# - Does the Haniyeh row show the darkest red across BTC/ETH?
#   (EMH: unanticipated events produce larger reactions)
# - Do later events show lighter colours than earlier ones?
#   (desensitisation: market learns to discount repeated shocks)
# - Feb 2026 row: may show green for USDC (surge, not drop)
#   suggesting different behavioral response at different market maturity

## Chart 3 — Correlation Scatter: USDC Volume vs BTC Price Change

Tests the core hypothesis: do on-chain payment flows and crypto prices move together during geopolitical shocks? Each point is one event. The regression line gives the Pearson r and p-value.

If p < 0.05, the correlation is statistically significant — unlikely to be due to chance with 9 data points.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

x = event_study['btc_d1'].dropna()
y = event_study.loc[x.index, 'usdc_d1']
labels = event_study.loc[x.index, 'event']

ax.scatter(x, y, s=100, color='steelblue', zorder=5)

for i, label in enumerate(labels):
    ax.annotate(
        label[:28],
        (x.iloc[i], y.iloc[i]),
        textcoords='offset points',
        xytext=(6, 4), fontsize=8
    )

slope, intercept, r, p, _ = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(
    x_line, slope * x_line + intercept,
    color='crimson', linewidth=1.5,
    linestyle='--',
    label=f'r = {r:.2f}, p = {p:.3f}'
)

ax.axhline(0, color='gray', linewidth=0.8)
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('BTC Price Change D+1 (%)')
ax.set_ylabel('USDC Volume Change D+1 (%)')
ax.set_title(
    'BTC Price vs USDC Payment Volume\n'
    'On Day of Conflict Escalation Events',
    fontweight='bold'
)
ax.legend()
plt.tight_layout()
plt.savefig('chart3_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Pearson r = {r:.3f}')
print(f'p-value   = {p:.3f}')
print(f'Slope     = {slope:.3f}')

# Interpretation guide:
# r > 0.6 and p < 0.05: strong significant positive correlation
#   -> USDC volume and BTC price move together during geopolitical shocks
# r < 0.3 or p > 0.05: weak / non-significant
#   -> payment behavior and price react independently
# Note: 9 data points is a small sample — interpret with appropriate caution.
# State findings as 'consistent with' rather than 'proves'.

## Desensitisation Test

Test whether absolute USDC volume reactions decrease over time — evidence of market desensitisation to repeated shocks of the same type (Berkman, Jacobsen & Lee, 2011).

In [ ]:
abs_usdc = event_study[['date', 'event', 'usdc_d1']].copy()
abs_usdc['abs_usdc_d1'] = abs_usdc['usdc_d1'].abs()
abs_usdc = abs_usdc.sort_values('date')

print('Absolute USDC Volume Change D+1 by Event (chronological):')
print(abs_usdc[['date', 'event', 'usdc_d1', 'abs_usdc_d1']].to_string())

# What to look for:
# If abs_usdc_d1 values trend downward from top to bottom of this table,
# that is evidence consistent with the desensitisation hypothesis:
# repeated exposure to conflict shocks reduced the behavioral response over time.
#
# Important: state this as 'consistent with' not 'proves' — sample size is 9.

## Economic Interpretation

### What the Results Tell Us

**USDC volume as a payment metric, not a price metric:**  
Unlike BTC or ETH prices — which measure speculative sentiment — USDC transfer volume measures real transactional economic activity. When it drops sharply on an event day, it means transactional demand collapsed: fewer actors were willing to initiate payments. In Keynesian terms, *liquidity preference increased* — holders chose to hold rather than transact (Keynes, 1936). This is flight-to-safety behavior visible in on-chain payment data.

**The October 7 finding:**  
A 66% collapse in USDC volume on the day of the Hamas attack, simultaneous with BTC and ETH price falls, suggests the shock affected not just speculative sentiment but actual transactional demand. Price reactions and payment behavior moved together.

**EMH and the Haniyeh assassination:**  
The Efficient Markets Hypothesis predicts that unanticipated events produce larger market reactions than anticipated ones (Fama, 1970). The Haniyeh assassination was unanticipated — an assassination of a senior Hamas leader inside Tehran, far outside the anticipated retaliation pattern. If this event shows the largest price and volume drops, that is evidence consistent with EMH applied to crypto payment flows.

**The February 2026 spike:**  
The USDC volume surge (>$600B/week) following the US-Israel strikes on Iran is the opposite of the October 2023 collapse. This may reflect panic capital *movement* rather than capital *freezing* — institutional actors rebalancing portfolios under a more mature market regime, rather than individual users halting transactions in shock.

**Limitation:**  
Nine data points is a small sample. All conclusions are stated as *consistent with* the relevant hypotheses, not as definitive proof. A larger event dataset — covering more geopolitical shocks across multiple conflict regions — would be needed for statistically robust conclusions.

---

### References

- Berkman, H., Jacobsen, B., & Lee, J. B. (2011). Time-varying rare disaster risk and stock returns. *Journal of Financial Economics*, 101(2), 313–332.
- Brown, S. J., & Warner, J. B. (1985). Using daily stock returns: The case of event studies. *Journal of Financial Economics*, 14(1), 3–31.
- Fama, E. F. (1970). Efficient capital markets: A review of theory and empirical work. *The Journal of Finance*, 25(2), 383–417.
- Keynes, J. M. (1936). *The general theory of employment, interest, and money.* Macmillan.
- Nakamoto, S. (2008). Bitcoin: A peer-to-peer electronic cash system. https://bitcoin.org/bitcoin.pdf